In [ ]:
import pandas as pd

CSV_PATH = "/content/loan.csv"

status_counts = {}

for chunk in pd.read_csv(
    CSV_PATH,
    usecols=["loan_status"],
    chunksize=50000
):
    counts = chunk["loan_status"].value_counts(dropna=False)

    for status, count in counts.items():
        status_counts[status] = status_counts.get(status, 0) + int(count)

print("Loan status counts in the original dataset:")
print(pd.Series(status_counts).sort_values(ascending=False))

Loan status counts in the original dataset:
Current               169505
Fully Paid              5682
Late (31-120 days)       972
In Grace Period          685
Late (16-30 days)        323
Charged Off              143
NaN                        1
dtype: int64


In [ ]:
import pandas as pd

CSV_PATH = "/content/loan.csv"
SAMPLE_PER_CLASS = 20000
CHUNK_SIZE = 50000

class_samples = {
    "Fully Paid": [],
    "Charged Off": []
}

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=True):
    chunk.columns = chunk.columns.str.strip()

    for status in class_samples:
        remaining = SAMPLE_PER_CLASS - sum(
            len(part) for part in class_samples[status]
        )

        if remaining <= 0:
            continue

        eligible = chunk[chunk["loan_status"] == status]

        if len(eligible) > 0:
            class_samples[status].append(
                eligible.sample(
                    n=min(len(eligible), remaining),
                    random_state=42
                )
            )

df = pd.concat(
    class_samples["Fully Paid"] + class_samples["Charged Off"],
    ignore_index=True
)

df = df.drop_duplicates()

df["target"] = (df["loan_status"] == "Charged Off").astype("int8")

print("Sample shape:", df.shape)
print(df["loan_status"].value_counts())
print(df["target"].value_counts())

/tmp/ipykernel_945/2191006302.py:12: DtypeWarning: Columns (123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=True):
/tmp/ipykernel_945/2191006302.py:12: DtypeWarning: Columns (123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=True):
/tmp/ipykernel_945/2191006302.py:12: DtypeWarning: Columns (139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=True):


Sample shape: (6316, 146)
loan_status
Fully Paid     6147
Charged Off     169
Name: count, dtype: int64
target
0    6147
1     169
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

target_col = "target"

leakage_keywords = [
    "loan_status", "recoveries", "collection_recovery_fee",
    "total_pymnt", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "last_pymnt", "next_pymnt",
    "out_prncp", "settlement", "hardship", "debt_settlement"
]

identifier_cols = [
    "id", "member_id", "url", "desc", "emp_title", "zip_code", "title"
]

excluded = [c for c in df.columns if
            c.lower().strip() in identifier_cols or
            any(k in c.lower().strip() for k in leakage_keywords)]

X = df.drop(columns=excluded + [target_col], errors="ignore")
y = df[target_col]

X = X.replace([np.inf, -np.inf], np.nan)

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=20,
            max_categories=30
        ))
    ]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Training data:", X_train_t.shape)
print("Testing data:", X_test_t.shape)
print("Training target distribution:")
print(y_train.value_counts())
print("Features after encoding:", len(feature_names))

/tmp/ipykernel_945/1416883086.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace([np.inf, -np.inf], np.nan)
/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['deferral_term' 'payment_plan_start_date'
 'orig_projected_additional_accrued_interest']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training data: (5052, 236)
Testing data: (1264, 236)
Training target distribution:
target
0    4917
1     135
Name: count, dtype: int64
Features after encoding: 236


/usr/local/lib/python3.13/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['deferral_term' 'payment_plan_start_date'
 'orig_projected_additional_accrued_interest']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np

X_mi = X_train_t.toarray() if hasattr(X_train_t, "toarray") else X_train_t

mi_scores = mutual_info_classif(
    X_mi,
    y_train,
    random_state=42
)

mi_results = pd.DataFrame({
    "Feature": feature_names,
    "MI_Score": mi_scores
})

mi_results = mi_results.sort_values(
    by="MI_Score",
    ascending=False
)

print("Top 20 Features by Mutual Information:")
print(mi_results.head(20).to_string(index=False))

Top 20 Features by Mutual Information:
                                 Feature  MI_Score
        cat__last_credit_pull_d_Feb-2019  0.006497
                      cat__addr_state_WA  0.006188
          cat__earliest_cr_line_Mar-2005  0.006058
                        num__open_rv_24m  0.005998
       cat__verification_status_Verified  0.005707
cat__earliest_cr_line_infrequent_sklearn  0.005668
                      cat__addr_state_VA  0.005557
                       cat__sub_grade_A3  0.005045
                                num__dti  0.004935
             num__mths_since_last_record  0.004780
                   cat__issue_d_Sep-2018  0.004698
                 cat__emp_length_8 years  0.004627
                       cat__sub_grade_A5  0.004485
                            num__bc_util  0.004233
                           num__int_rate  0.004216
                      cat__addr_state_IL  0.004137
                   cat__issue_d_Dec-2018  0.004035
                    num__revol_bal_joint  0

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
from scipy import sparse

X_mi = X_train_t

if sparse.issparse(X_mi):
    X_mi = X_mi.toarray()

mi_scores = mutual_info_classif(
    X_mi,
    y_train,
    random_state=42
)

mi_results = pd.DataFrame({
    "Feature": feature_names,
    "MI_Score": mi_scores
}).sort_values("MI_Score", ascending=False)

print(mi_results.head(20).to_string(index=False))

                                 Feature  MI_Score
        cat__last_credit_pull_d_Feb-2019  0.006497
                      cat__addr_state_WA  0.006188
          cat__earliest_cr_line_Mar-2005  0.006058
                        num__open_rv_24m  0.005998
       cat__verification_status_Verified  0.005707
cat__earliest_cr_line_infrequent_sklearn  0.005668
                      cat__addr_state_VA  0.005557
                       cat__sub_grade_A3  0.005045
                                num__dti  0.004935
             num__mths_since_last_record  0.004780
                   cat__issue_d_Sep-2018  0.004698
                 cat__emp_length_8 years  0.004627
                       cat__sub_grade_A5  0.004485
                            num__bc_util  0.004233
                           num__int_rate  0.004216
                      cat__addr_state_IL  0.004137
                   cat__issue_d_Dec-2018  0.004035
                    num__revol_bal_joint  0.003922
          cat__earliest_cr_line

In [ ]:
print("Dataset:", df.shape)
print("Training features:", X_train_t.shape)
print("Testing features:", X_test_t.shape)
print("Number of encoded features:", len(feature_names))
print("Training target counts:")
print(y_train.value_counts())

Dataset: (6316, 146)
Training features: (5052, 236)
Testing features: (1264, 236)
Number of encoded features: 236
Training target counts:
target
0    4917
1     135
Name: count, dtype: int64


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

K = min(20, X_train_t.shape[1])

mi = SelectKBest(mutual_info_classif, k=K)
mi.fit(X_train_t, y_train)

anova = SelectKBest(f_classif, k=K)
anova.fit(X_train_t, y_train)

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42,
    n_jobs=2
)
xgb.fit(X_train_t, y_train)

gain = xgb.feature_importances_
gain_idx = np.argsort(gain)[-K:]

methods = {
    "Mutual Information": mi.get_support(),
    "ANOVA F-test": anova.get_support()
}

methods["XGBoost Gain"] = np.isin(
    np.arange(len(feature_names)), gain_idx
)

results = []

for name, mask in methods.items():
    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=2
    )

    model.fit(X_train_t[:, mask], y_train)
    probs = model.predict_proba(X_test_t[:, mask])[:, 1]

    results.append({
        "Method": name,
        "Features_Selected": int(mask.sum()),
        "Test_ROC_AUC": roc_auc_score(y_test, probs),
        "Selected_Features": ", ".join(feature_names[mask][:10])
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 18  21  64  65  71 145] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


            Method  Features_Selected  Test_ROC_AUC                                                                                                                                                                                           Selected_Features
Mutual Information                 20      0.720361                   num__annual_inc, num__dti, num__policy_code, num__open_acc_6m, num__open_il_24m, num__acc_open_past_24mths, num__bc_util, num__mort_acc, num__num_accts_ever_120_pd, num__tot_hi_cred_lim
      ANOVA F-test                 20      0.788965 num__int_rate, num__total_acc, num__acc_open_past_24mths, num__num_sats, cat__grade_A, cat__grade_E, cat__grade_infrequent_sklearn, cat__sub_grade_E4, cat__sub_grade_E5, cat__sub_grade_infrequent_sklearn
      XGBoost Gain                 20      0.757221              num__funded_amnt_inv, num__int_rate, num__annual_inc, num__inq_last_6mths, num__mths_since_last_record, num__total_acc, num__tot_cur_bal, num__open_acc_6m, num__open_i

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

X_eval = X_test_t
y_eval = y_test

if X_eval.shape[0] > 3000:
    rng = np.random.RandomState(42)
    idx = rng.choice(X_eval.shape[0], 3000, replace=False)
    X_eval = X_eval[idx]
    y_eval = y_eval.iloc[idx]

perm = permutation_importance(
    xgb,
    X_eval,
    y_eval,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=2
)

perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Importance": perm.importances_mean,
    "Std_Importance": perm.importances_std
}).sort_values("Mean_Importance", ascending=False)

print("Top 20 permutation-importance features:")
print(perm_df.head(20).to_string(index=False))

print("\nMean standard deviation:",
      perm_df["Std_Importance"].mean())

print("Features with negative mean importance:",
      (perm_df["Mean_Importance"] < 0).sum())

Top 20 permutation-importance features:
                            Feature  Mean_Importance  Std_Importance
   cat__last_credit_pull_d_Feb-2019         0.174821        0.031299
              cat__issue_d_Dec-2018         0.030832        0.009197
              cat__issue_d_Sep-2018         0.026719        0.003708
              cat__issue_d_Aug-2018         0.020330        0.010898
                      num__int_rate         0.018039        0.004688
                       num__bc_util         0.017508        0.007219
              cat__issue_d_Nov-2018         0.013773        0.007140
              cat__issue_d_Oct-2018         0.011028        0.010865
num__mths_since_recent_revol_delinq         0.006887        0.004780
          cat__emp_length_10+ years         0.006260        0.003594
   cat__last_credit_pull_d_Jan-2019         0.006126        0.002874
      cat__disbursement_method_Cash         0.005375        0.001296
                           num__dti         0.005108        0.0

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd
import numpy as np

l1_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=0.1,
    max_iter=1000,
    random_state=42
)

l2_model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=0.1,
    max_iter=500,
    random_state=42
)

l1_model.fit(X_train_t, y_train)
l2_model.fit(X_train_t, y_train)

models = {
    "L1 Regularization": l1_model,
    "L2 Regularization": l2_model
}

results = []

for name, model in models.items():
    probabilities = model.predict_proba(X_test_t)[:, 1]
    predictions = model.predict(X_test_t)
    coefficients = model.coef_[0]

    results.append({
        "Method": name,
        "Nonzero_Features": int(np.sum(np.abs(coefficients) > 1e-8)),
        "ROC_AUC": roc_auc_score(y_test, probabilities),
        "Accuracy": accuracy_score(y_test, predictions)
    })

regularization_results = pd.DataFrame(results)

print(regularization_results.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


           Method  Nonzero_Features  ROC_AUC  Accuracy
L1 Regularization                37 0.750598  0.973101
L2 Regularization               231 0.724534  0.973101


In [ ]:
coef_results = pd.DataFrame({
    "Feature": feature_names,
    "L1_Coefficient": l1_model.coef_[0],
    "L2_Coefficient": l2_model.coef_[0]
})

print("Top L1 features:")
print(
    coef_results.assign(
        Magnitude=coef_results["L1_Coefficient"].abs()
    ).sort_values("Magnitude", ascending=False)
    .head(15).to_string(index=False)
)

print("\nTop L2 features:")
print(
    coef_results.assign(
        Magnitude=coef_results["L2_Coefficient"].abs()
    ).sort_values("Magnitude", ascending=False)
    .head(15).to_string(index=False)
)

Top L1 features:
                              Feature  L1_Coefficient  L2_Coefficient  Magnitude
     cat__last_credit_pull_d_Feb-2019        1.010496        1.020128   1.010496
                cat__issue_d_Aug-2018        0.458209        0.733784   0.458209
                cat__issue_d_Dec-2018       -0.453660       -0.626841   0.453660
                        num__int_rate        0.408648        0.456497   0.408648
                cat__issue_d_Sep-2018        0.398696        0.629013   0.398696
         cat__home_ownership_MORTGAGE       -0.259809       -0.195513   0.259809
                cat__issue_d_Nov-2018       -0.249835       -0.427816   0.249835
                cat__issue_d_Oct-2018       -0.186470       -0.314483   0.186470
                        num__all_util       -0.163047       -0.380880   0.163047
                    cat__pymnt_plan_n       -0.145164       -0.006343   0.145164
cat__verification_status_Not Verified       -0.137712       -0.222361   0.137712
           

In [ ]:
import pandas as pd
import numpy as np

booster = xgb.get_booster()

gain_scores = booster.get_score(importance_type="gain")
weight_scores = booster.get_score(importance_type="weight")

gain_values = []
weight_values = []

for i in range(len(feature_names)):
    key = f"f{i}"
    gain_values.append(gain_scores.get(key, 0))
    weight_values.append(weight_scores.get(key, 0))

xgb_importance = pd.DataFrame({
    "Feature": feature_names,
    "Gain": gain_values,
    "Weight": weight_values
})

xgb_importance["Gain_Rank"] = (
    xgb_importance["Gain"].rank(ascending=False, method="min")
)

xgb_importance["Weight_Rank"] = (
    xgb_importance["Weight"].rank(ascending=False, method="min")
)

print("Top 20 Features by Gain:")
print(
    xgb_importance.sort_values("Gain", ascending=False)
    .head(20).to_string(index=False)
)

print("\nTop 20 Features by Weight:")
print(
    xgb_importance.sort_values("Weight", ascending=False)
    .head(20).to_string(index=False)
)

Top 20 Features by Gain:
                         Feature      Gain  Weight  Gain_Rank  Weight_Rank
           cat__issue_d_Sep-2018 24.964289    12.0        1.0         31.0
           cat__issue_d_Aug-2018 20.633018    13.0        2.0         25.0
cat__last_credit_pull_d_Feb-2019 20.457338    29.0        3.0          9.0
                   num__int_rate 12.552796    44.0        4.0          2.0
                num__open_il_24m 11.955425     3.0        5.0         59.0
            num__funded_amnt_inv 11.445854     3.0        6.0         59.0
                    cat__grade_E  8.256237     1.0        7.0         77.0
                num__open_acc_6m  8.115870    11.0        8.0         35.0
          num__total_bal_ex_mort  8.100513    16.0        9.0         19.0
     num__mths_since_last_record  8.059782     1.0       10.0         77.0
                num__open_rv_12m  7.767566     8.0       11.0         45.0
                  num__total_acc  7.543128    23.0       12.0         13.0


In [ ]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(
    xgb_importance["Gain_Rank"],
    xgb_importance["Weight_Rank"]
)

gain_top = set(
    xgb_importance.nlargest(20, "Gain")["Feature"]
)

weight_top = set(
    xgb_importance.nlargest(20, "Weight")["Feature"]
)

print("Spearman rank correlation:", round(rho, 4))
print("P-value:", p_value)
print("Common top-20 features:", len(gain_top & weight_top))
print("Gain-only top-20 features:", len(gain_top - weight_top))
print("Weight-only top-20 features:", len(weight_top - gain_top))

Spearman rank correlation: 0.9555
P-value: 4.4905729102032285e-126
Common top-20 features: 7
Gain-only top-20 features: 13
Weight-only top-20 features: 13


In [ ]:
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

seeds = [21, 42, 63]
gain_results = []

for seed in seeds:
    rng = np.random.RandomState(seed)

    idx = rng.choice(
        X_train_t.shape[0],
        size=min(10000, X_train_t.shape[0]),
        replace=False
    )

    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=seed,
        n_jobs=2
    )

    model.fit(X_train_t[idx], y_train.iloc[idx])

    scores = model.get_booster().get_score(
        importance_type="gain"
    )

    gain_results.append([
        scores.get(f"f{i}", 0)
        for i in range(len(feature_names))
    ])

gain_df = pd.DataFrame(
    np.array(gain_results).T,
    index=feature_names,
    columns=[f"Run_{seed}" for seed in seeds]
)

rank_df = gain_df.rank(
    axis=0,
    ascending=False,
    method="average"
)

print("Spearman rank correlation:")
print(rank_df.corr(method="spearman").round(4))

top_features = {
    col: set(gain_df[col].nlargest(20).index)
    for col in gain_df.columns
}

print("\nTop-20 feature overlap:")
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        common = top_features[f"Run_{seeds[i]}"] & top_features[f"Run_{seeds[j]}"]
        print(
            f"Run {seeds[i]} vs Run {seeds[j]}: "
            f"{len(common)}/20 common features"
        )

print("\nMean gain across runs:")
print(
    gain_df.mean(axis=1)
    .sort_values(ascending=False)
    .head(20)
)

Spearman rank correlation:
        Run_21  Run_42  Run_63
Run_21  1.0000  0.9105  0.9269
Run_42  0.9105  1.0000  0.9318
Run_63  0.9269  0.9318  1.0000

Top-20 feature overlap:
Run 21 vs Run 42: 15/20 common features
Run 21 vs Run 63: 15/20 common features
Run 42 vs Run 63: 16/20 common features

Mean gain across runs:
cat__last_credit_pull_d_Feb-2019            21.612552
cat__issue_d_Sep-2018                       21.448913
cat__issue_d_Aug-2018                       20.111678
num__int_rate                               12.426286
num__funded_amnt_inv                        11.471026
num__open_il_24m                            10.552646
num__num_il_tl                               8.641767
num__total_bal_ex_mort                       8.269624
num__annual_inc                              8.142839
num__inq_last_6mths                          7.965005
cat__issue_d_Oct-2018                        7.759612
cat__verification_status_Source Verified     7.755240
num__open_acc_6m                

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

summary = []

summary.append({
    "Observation": "1. Feature Selection Methods",
    "Result": results_df[["Method", "Test_ROC_AUC"]].to_dict("records")
})

summary.append({
    "Observation": "2. Permutation Importance",
    "Result": f"Mean importance SD: {perm_df['Std_Importance'].mean():.6f}"
})

summary.append({
    "Observation": "3. L1 vs L2 Regularization",
    "Result": regularization_results.to_dict("records")
})

summary.append({
    "Observation": "4. XGBoost Gain vs Weight",
    "Result": f"Rank correlation: {rho:.4f}; Top-20 overlap: {len(gain_top & weight_top)}"
})

summary.append({
    "Observation": "5. Gain Stability",
    "Result": rank_df.corr(method="spearman").round(4).to_dict()
})

for item in summary:
    print("\n", item["Observation"])
    print(item["Result"])


 1. Feature Selection Methods
[{'Method': 'Mutual Information', 'Test_ROC_AUC': 0.7203610712577714}, {'Method': 'ANOVA F-test', 'Test_ROC_AUC': 0.7889646102343375}, {'Method': 'XGBoost Gain', 'Test_ROC_AUC': 0.757221425155428}]

 2. Permutation Importance
Mean importance SD: 0.000903

 3. L1 vs L2 Regularization
[{'Method': 'L1 Regularization', 'Nonzero_Features': 37, 'ROC_AUC': 0.750597800095648, 'Accuracy': 0.9731012658227848}, {'Method': 'L2 Regularization', 'Nonzero_Features': 231, 'ROC_AUC': 0.7245337159253945, 'Accuracy': 0.9731012658227848}]

 4. XGBoost Gain vs Weight
Rank correlation: 0.9555; Top-20 overlap: 7

 5. Gain Stability
{'Run_21': {'Run_21': 1.0, 'Run_42': 0.9105, 'Run_63': 0.9269}, 'Run_42': {'Run_21': 0.9105, 'Run_42': 1.0, 'Run_63': 0.9318}, 'Run_63': {'Run_21': 0.9269, 'Run_42': 0.9318, 'Run_63': 1.0}}


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

X_train, X_test = X_train.align(
    X_test, join="left", axis=1, fill_value=0
)

X_train.columns = [f"feature_{i}" for i in range(X_train.shape[1])]
X_test.columns = X_train.columns

xgb.fit(X_train, y_train)

xgb_predictions = xgb.predict(X_test)
xgb_probabilities = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Test Performance")
print("-" * 35)

print(f"Accuracy:  {accuracy_score(y_test, xgb_predictions):.4f}")
print(f"Precision: {precision_score(y_test, xgb_predictions, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, xgb_predictions, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, xgb_predictions, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, xgb_probabilities):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    xgb_predictions,
    target_names=["Fully Paid", "Charged Off"],
    zero_division=0
))

XGBoost Test Performance
-----------------------------------
Accuracy:  0.9731
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000
ROC-AUC:   0.7726

Confusion Matrix:
[[1230    0]
 [  34    0]]

Classification Report:
              precision    recall  f1-score   support

  Fully Paid       0.97      1.00      0.99      1230
 Charged Off       0.00      0.00      0.00        34

    accuracy                           0.97      1264
   macro avg       0.49      0.50      0.49      1264
weighted avg       0.95      0.97      0.96      1264

